In [1]:
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
#from imblearn.over_sampling import SMOTE
#from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB
import joblib


Note: This version does not perform CV for the positive data. Applies 5-fold undersampling for neg data and saves each model

In [5]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [6]:
root_folder='../data/train_data/embedding/birdnet/'

In [8]:
df_pos = get_embedding_birdnet(root_folder, 1)
df_neg = get_embedding_birdnet(root_folder, 0)

In [9]:
df_neg["filename"] = df_neg["embed_name"].str.split("_").str[0]
df_pos["filename"] = df_pos["embed_name"].str.split("-").str[0].str[:-1]

In [10]:
df_pos['target'] = 1
df_neg['target'] = 0

In [12]:
df_neg.sample(3)

,embed_name,embedding,filename,target
1886,PxU8xj_1792.birdnet.embeddings.txt,"[0.0, 0.08188188, 0.0, 0.07759682, 0.23009986,...",PxU8xj,0
760,IagluS_1348.birdnet.embeddings.txt,"[0.09201571, 0.19385593, 0.29428566, 0.0, 0.18...",IagluS,0
913,IlaTbo_1088.birdnet.embeddings.txt,"[0.0, 0.15067746, 1.223087, 0.53139, 0.995378,...",IlaTbo,0


Perform 5-fold split for the negative data only

In [13]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Create a new column to store fold numbers
df_neg["fold"] = -1  # Initialize with -1

for fold, (train_idx, test_idx) in enumerate(kf.split(df_neg)):
    df_neg.loc[test_idx, "fold"] = fold  # Assign fold number to test samples

SVM

In [ ]:
import joblib
from sklearn.svm import SVC

# Run for all 5 folds
for fold in range(5):

    print(f"\nTraining model for fold {fold}...")

    # Select negatives for this fold
    df_neg_fold = df_neg[df_neg["fold"] == fold].copy()
    df_neg_fold = df_neg_fold.drop("fold", axis=1)

    # Combine positives and negatives
    df_combined = pd.concat(
        [df_pos, df_neg_fold],
        ignore_index=True
    )

    # Shuffle
    df_combined = df_combined.sample(
        frac=1,
        random_state=22
    ).reset_index(drop=True)

    # Features and labels
    X = np.vstack(df_combined["embedding"].values)
    y = df_combined["target"].values

    # Create SVC
    model = SVC(
        kernel="rbf",
        probability=True,
        random_state=22
    )

    # Train on full dataset for this fold
    model.fit(X, y)

    # Save model
    model_path = f"svc_model_negfold_{fold}.joblib"
    joblib.dump(model, model_path)

    print(f"Model saved as: {model_path}")

print("\nAll 5 models trained and saved.")




Training model for fold 0...
Model saved as: svc_model_negfold_0.joblib

Training model for fold 1...
Model saved as: svc_model_negfold_1.joblib

Training model for fold 2...
Model saved as: svc_model_negfold_2.joblib

Training model for fold 3...
Model saved as: svc_model_negfold_3.joblib

Training model for fold 4...
Model saved as: svc_model_negfold_4.joblib

All 5 models trained and saved.


In [15]:
train_accuracy = model.score(X, y)
print(f"Train accuracy: {train_accuracy:.4f}")

Train accuracy: 1.0000


Random Forest

In [ ]:
# Run for all 5 negative folds
for fold in range(5):

    print(f"\nTraining model for fold {fold}...")

    # Select negatives for this fold
    df_neg_fold = df_neg[df_neg["fold"] == fold].copy()
    df_neg_fold = df_neg_fold.drop("fold", axis=1)

    # Combine positives and negatives
    df_combined = pd.concat(
        [df_pos, df_neg_fold],
        ignore_index=True
    )

    # Shuffle
    df_combined = df_combined.sample(
        frac=1,
        random_state=22
    ).reset_index(drop=True)

    # Features and labels
    X = np.vstack(df_combined["embedding"].values)
    y = df_combined["target"].values

    # Create RFC
    model = RFC(n_jobs = -1)

    # Train on full dataset for this fold
    model.fit(X, y)

    # Save model
    model_path = f"rfc_model_negfold_{fold}.joblib"
    joblib.dump(model, model_path)

    print(f"Model saved as: {model_path}")

print("\nAll 5 models trained and saved.")


Training model for fold 0...
Model saved as: rfc_model_negfold_0.joblib

Training model for fold 1...
Model saved as: rfc_model_negfold_1.joblib

Training model for fold 2...
Model saved as: rfc_model_negfold_2.joblib

Training model for fold 3...
Model saved as: rfc_model_negfold_3.joblib

Training model for fold 4...
Model saved as: rfc_model_negfold_4.joblib

All 5 models trained and saved.


XGBoost

In [17]:


# Run for all 5 negative folds
for fold in range(5):

    print(f"\nTraining model for fold {fold}...")

    # Select negatives for this fold
    df_neg_fold = df_neg[df_neg["fold"] == fold].copy()
    df_neg_fold = df_neg_fold.drop("fold", axis=1)

    # Combine positives and negatives
    df_combined = pd.concat(
        [df_pos, df_neg_fold],
        ignore_index=True
    )

    # Shuffle
    df_combined = df_combined.sample(
        frac=1,
        random_state=22
    ).reset_index(drop=True)

    # Features and labels
    X = np.vstack(df_combined["embedding"].values)
    y = df_combined["target"].values

    # Create RFC
    model = XGB.XGBClassifier(objective='binary:logistic')

    # Train on full dataset for this fold
    model.fit(X, y)

    # Save model
    model_path = f"xgb_model_negfold_{fold}.joblib"
    joblib.dump(model, model_path)

    print(f"Model saved as: {model_path}")



Training model for fold 0...
Model saved as: xgb_model_negfold_0.joblib

Training model for fold 1...
Model saved as: xgb_model_negfold_1.joblib

Training model for fold 2...
Model saved as: xgb_model_negfold_2.joblib

Training model for fold 3...
Model saved as: xgb_model_negfold_3.joblib

Training model for fold 4...
Model saved as: xgb_model_negfold_4.joblib


This part is from the old script, where 5-fold CV is done on the positives as well

In [12]:
import os
import joblib
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.metrics import confusion_matrix

cv = GroupKFold(n_splits=5)

dataset = "ds1"

accuracies = []
aps = []
confusion_matrices = []

os.makedirs("saved_models", exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = SVC(kernel="rbf", cache_size=500)

    model.fit(X_train, y_train)

    # Save model
    joblib.dump(model, f"saved_models/{dataset}_svm_fold_{fold}.joblib")

    # Evaluate
    y_pred = model.predict(X_test)
    y_score = model.decision_function(X_test)

    accuracies.append(accuracy_score(y_test, y_pred))
    aps.append(average_precision_score(y_test, y_score))
    cm = confusion_matrix(y_test, y_pred)
    confusion_matrices.append(cm)

    np.savetxt(
        f"saved_models/confusion_matrix_fold_{fold}.csv",
        cm,
        fmt="%d",
        delimiter=","
    )

print("Accuracy per fold:", np.round(accuracies, 4))
print(f"Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies, ddof=1):.4f}")

print("AP per fold:", np.round(aps, 4))
print(f"AP: {np.mean(aps):.4f} ± {np.std(aps, ddof=1):.4f}")


Accuracy per fold: [0.9801 0.9801 0.97   0.98   0.995 ]
Accuracy: 0.9810 ± 0.0089
AP per fold: [0.9993 0.9997 0.9973 0.9979 0.9999]
AP: 0.9988 ± 0.0011


In [60]:
#plt.plot(recall, precision, marker='.')
#plt.xlabel('Recall')
#plt.ylabel('Precision')
#plt.title('Precision-Recall Curve')
#plt.show()